# 17 — Asyncio: `async` / `await` and Cooperative Concurrency

Goal: write high-throughput I/O-bound code using the asyncio event loop.

_Generated: 2026-02-19_

## Setup

This course targets **Python 3.11+** (works on 3.10+, with a few feature differences).

Recommended tooling:

```bash
# create + activate a virtual environment
python -m venv .venv
# mac/linux:
source .venv/bin/activate
# windows (PowerShell):
# .venv\Scripts\Activate.ps1

python -m pip install -U pip

# quality-of-life (optional but recommended)
python -m pip install -U ipykernel ruff black pytest mypy
python -m pip install -U httpx aiohttp
```

If you're using Jupyter:
```bash
python -m ipykernel install --user --name python-course --display-name "Python Course (.venv)"
```

In [ ]:

import sys, platform, os
print("python:", sys.version.split()[0])
print("implementation:", platform.python_implementation())
print("platform:", platform.platform())
print("cwd:", os.getcwd())


## 1.
L1: Mental model

Asyncio is single-threaded by default.
Concurrency comes from *awaiting* I/O and letting other tasks run.

Key terms:
- coroutine (`async def`)
- task (`asyncio.create_task`)
- event loop

In [ ]:

import asyncio

async def say(msg: str, delay: float):
    await asyncio.sleep(delay)
    return msg

async def main():
    a = asyncio.create_task(say("A", 0.05))
    b = asyncio.create_task(say("B", 0.01))
    done = await asyncio.gather(a, b)
    print(done)

asyncio.run(main())


## 2.
L2: Timeouts and cancellation

Cancellation is part of normal async control flow.

In [ ]:

import asyncio

async def slow():
    await asyncio.sleep(1)
    return "ok"

async def main():
    try:
        return await asyncio.wait_for(slow(), timeout=0.05)
    except asyncio.TimeoutError:
        return "timed out"

print(asyncio.run(main()))


## 3.
L3: Async queues (producer/consumer)

Async queues coordinate coroutines safely.

In [ ]:

import asyncio

async def producer(q: asyncio.Queue[int]):
    for i in range(5):
        await q.put(i)
    await q.put(-1)  # sentinel

async def consumer(q: asyncio.Queue[int]) -> list[int]:
    out: list[int] = []
    while True:
        x = await q.get()
        if x == -1:
            break
        out.append(x * x)
    return out

async def main():
    q: asyncio.Queue[int] = asyncio.Queue()
    prod = asyncio.create_task(producer(q))
    out = await consumer(q)
    await prod
    print(out)

asyncio.run(main())


## 4.
L4: When to use asyncio vs threads

Use asyncio when:
- you have many I/O operations (HTTP, websockets, DB calls)
- you want high concurrency with low overhead

Use threads when:
- libraries are blocking and you can’t change them
- you need parallelism for I/O but don’t want async refactors

## 5.
L5: Exercises

1. Write an async function that runs 100 small sleeps concurrently and times it.
2. Add cancellation handling to a long-running coroutine.
3. (Optional) Use `httpx.AsyncClient` to fetch multiple pages concurrently.

## 6.
L6: Concurrency limits with `Semaphore`

Never launch unbounded tasks against a real API.

In [ ]:

import asyncio

async def limited_worker(i: int, sem: asyncio.Semaphore):
    async with sem:
        await asyncio.sleep(0.02)
        return i

async def main():
    sem = asyncio.Semaphore(5)
    tasks = [asyncio.create_task(limited_worker(i, sem)) for i in range(20)]
    out = await asyncio.gather(*tasks)
    print("done:", len(out))

asyncio.run(main())


## 7.
L7: Bridging blocking code: `asyncio.to_thread`

If you must call blocking functions from async code, run them in a thread.

In [ ]:

import asyncio, time

def blocking():
    time.sleep(0.05)
    return "ok"

async def main():
    return await asyncio.to_thread(blocking)

print(asyncio.run(main()))
